In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from matplotlib import font_manager
font_path = '/content/THSarabunNew.ttf'
font_manager.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'TH Sarabun New'

df = pd.read_csv("reviwe.csv")

# ถ้าไม่มี "คะแนน" ให้สร้างจากค่าเฉลี่ย
if 'คะแนน' not in df.columns:
    df['คะแนน'] = df[['ห้องพัก','บริการ','สถานที่ตั้ง']].mean(axis=1).round()

# -----------------------------
# 2) Sentiment Score เบื้องต้น
# -----------------------------
pos_words = ["good","great","excellent","amazing","friendly","clean","spotless","professional","helpful","comfy","perfect","nice","beautiful"]
neg_words = ["bad","terrible","awful","dirty","noisy","slow","rude","unfriendly","poor","broken","smelly","small","uncomfortable"]

def sentiment_score(text):
    if not isinstance(text, str):
        return 0
    tokens = re.findall(r"[A-Za-z']+", text.lower())
    pos = sum(t in pos_words for t in tokens)
    neg = sum(t in neg_words for t in tokens)
    if pos+neg == 0:
        return 0
    return (pos-neg)/(pos+neg)

df["sentiment"] = df["text"].apply(sentiment_score)

# -----------------------------
# 3) โมเดล Decision Tree
# -----------------------------
X = df[['บริการ','ห้องพัก','สถานที่ตั้ง','sentiment','platform']]
y = df['คะแนน']

preprocess = ColumnTransformer([
    ('num','passthrough',['บริการ','ห้องพัก','สถานที่ตั้ง','sentiment']),
    ('cat',OneHotEncoder(handle_unknown="ignore"),['platform'])
])

model = DecisionTreeRegressor(max_depth=5, random_state=42)
pipe = Pipeline([("preprocess", preprocess), ("model", model)])

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
pipe.fit(X_train,y_train)

# -----------------------------
# 4) Feature Importance
# -----------------------------
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
ohe_names = list(ohe.get_feature_names_out(['platform']))
all_names = ['บริการ','ห้องพัก','สถานที่ตั้ง','sentiment'] + ohe_names
importances = pipe.named_steps["model"].feature_importances_

# รวม Platform
rows, platform_total = [], 0
for n, imp in zip(all_names, importances):
    if n.startswith("platform_"):
        platform_total += imp
    else:
        rows.append([n, imp])
rows.append(["Platform", platform_total])

imp_df = pd.DataFrame(rows, columns=["Feature","Importance"])
imp_df["Importance %"] = (imp_df["Importance"]/imp_df["Importance"].sum())*100
imp_df = imp_df.sort_values("Importance %", ascending=False)

# export feature importance
imp_df.to_csv("feature_importance.csv", index=False)

# plot importance
plt.figure(figsize=(6,4))
plt.barh(imp_df["Feature"], imp_df["Importance %"])
plt.gca().invert_yaxis()
plt.title("Feature Importance (%)")
plt.xlabel("%")
plt.savefig("feature_importance_plot.png")
plt.close()

# decision tree plot
plt.figure(figsize=(12,6))
plot_tree(pipe.named_steps["model"], feature_names=all_names, filled=True, fontsize=6)
plt.savefig("decision_tree_plot.png")
plt.close()

# -----------------------------
# 5) Sentiment Analysis Summary
# -----------------------------
sentiment_summary = df["sentiment"].apply(
    lambda x: "positive" if x>0 else ("negative" if x<0 else "neutral")
).value_counts(normalize=True)

positive = round(sentiment_summary.get("positive",0)*100,1)
negative = round(sentiment_summary.get("negative",0)*100,1)
neutral  = round(sentiment_summary.get("neutral",0)*100,1)

# export sentiment results
df.to_csv("sentiment_analysis_results.csv", index=False)

print("\n=== Feature Importance ===")
print(imp_df)


print("\n📈 KPIs ที่ควรติดตาม")
print(" - คะแนนความพึงพอใจเฉลี่ย (เป้าหมาย: ⭐⭐⭐⭐ ขึ้นไป)")
print(" - อัตราความรู้สึกบวก (Positive Sentiment > 60%)")
print(" - การลดลงของคำเชิงลบที่พบบ่อย (ลดลงต่อเนื่องในแต่ละไตรมาส)")
print(" - Response Rate ของทีมงานโรงแรม (ตอบรีวิว > 90%)\n")

print("📁 ไฟล์ Output ที่สร้าง")
print(" - feature_importance.csv → ตารางสรุปปัจจัยและสัดส่วนความสำคัญ (%)")
print(" - decision_tree_plot.png → แผนภาพ Decision Tree")
print(" - feature_importance_plot.png → กราฟแท่ง Feature Importance")
print(" - sentiment_analysis_results.csv → สรุปผลวิเคราะห์ข้อความรีวิว\n")

print("📊 สถิติสุดท้าย")
print(f" 📝 จำนวนรีวิวทั้งหมด: {len(df)}")
print(f" 😊 สัดส่วนรีวิวเชิงบวก: ~{positive}%")
print(f" 😞 สัดส่วนรีวิวเชิงลบ: ~{negative}%")
print(f" 😐 สัดส่วนรีวิวกลาง: ~{neutral}%")
print(f" 📈 คะแนนเฉลี่ย (จากโมเดล): ~{round(df['คะแนน'].mean()/10,2)} / 10")
